# RAG Capacity Calculator

Calculate storage and usage estimates for your student notes RAG system.

In [ ]:
# Configuration
EMBEDDING_DIMENSION = 768  # Gemini with output_dimensionality=768
BYTES_PER_DIMENSION = 4    # Float32
CHUNK_SIZE_TOKENS = 400    # Average tokens per chunk
CHUNK_OVERLAP_TOKENS = 50  # Overlap between chunks

# Pinecone Free Tier Limits
MAX_STORAGE_GB = 2
MAX_WRITE_UNITS_MONTHLY = 2_000_000
MAX_READ_UNITS_MONTHLY = 1_000_000
MAX_NAMESPACES = 100

# Estimates
TOKENS_PER_PAGE = 500      # Average for academic text
QUERIES_PER_STUDENT_DAILY = 10

In [ ]:
def calculate_chunks_per_document(pages: int) -> int:
    """Calculate number of chunks for a document"""
    total_tokens = pages * TOKENS_PER_PAGE
    effective_chunk_size = CHUNK_SIZE_TOKENS - CHUNK_OVERLAP_TOKENS
    chunks = (total_tokens + effective_chunk_size - 1) // effective_chunk_size
    return chunks

def calculate_storage_per_vector() -> float:
    """Calculate storage in MB per vector"""
    bytes_per_vector = EMBEDDING_DIMENSION * BYTES_PER_DIMENSION
    mb_per_vector = bytes_per_vector / (1024 * 1024)
    return mb_per_vector

def calculate_capacity(students: int, pages_per_student: int):
    """Calculate total capacity requirements"""
    # Chunks calculation
    chunks_per_student = calculate_chunks_per_document(pages_per_student)
    total_chunks = students * chunks_per_student
    
    # Storage calculation
    mb_per_vector = calculate_storage_per_vector()
    total_storage_mb = total_chunks * mb_per_vector
    total_storage_gb = total_storage_mb / 1024
    
    # Write units (1 WU per vector upsert)
    write_units = total_chunks
    
    # Read units (queries * vectors returned)
    daily_queries = students * QUERIES_PER_STUDENT_DAILY
    monthly_queries = daily_queries * 30
    read_units = monthly_queries * 8  # Assuming top_k=8
    
    # Free tier usage
    storage_usage_pct = (total_storage_gb / MAX_STORAGE_GB) * 100
    write_usage_pct = (write_units / MAX_WRITE_UNITS_MONTHLY) * 100
    read_usage_pct = (read_units / MAX_READ_UNITS_MONTHLY) * 100
    namespace_usage_pct = (students / MAX_NAMESPACES) * 100
    
    return {
        'students': students,
        'pages_per_student': pages_per_student,
        'chunks_per_student': chunks_per_student,
        'total_chunks': total_chunks,
        'storage_mb': total_storage_mb,
        'storage_gb': total_storage_gb,
        'write_units': write_units,
        'read_units_monthly': read_units,
        'storage_usage_pct': storage_usage_pct,
        'write_usage_pct': write_usage_pct,
        'read_usage_pct': read_usage_pct,
        'namespace_usage_pct': namespace_usage_pct,
        'within_free_tier': all([
            storage_usage_pct <= 100,
            write_usage_pct <= 100,
            read_usage_pct <= 100,
            namespace_usage_pct <= 100
        ])
    }

In [ ]:
# Interactive calculator
print("=== RAG Capacity Calculator ===")
print()

# Example scenarios
scenarios = [
    (10, 100),    # Small class
    (50, 200),    # Medium class
    (100, 150),   # Large class
    (200, 100),   # Very large class
]

for students, pages in scenarios:
    result = calculate_capacity(students, pages)
    
    print(f"\nScenario: {students} students, {pages} pages each")
    print(f"  Total chunks: {result['total_chunks']:,}")
    print(f"  Storage: {result['storage_gb']:.2f} GB ({result['storage_usage_pct']:.1f}% of free tier)")
    print(f"  Write units: {result['write_units']:,} ({result['write_usage_pct']:.1f}% of free tier)")
    print(f"  Read units/month: {result['read_units_monthly']:,} ({result['read_usage_pct']:.1f}% of free tier)")
    print(f"  Namespaces: {students} ({result['namespace_usage_pct']:.1f}% of free tier)")
    print(f"  Within free tier: {'✅ Yes' if result['within_free_tier'] else '❌ No'}")

In [ ]:
# Custom calculation
print("\n=== Custom Calculation ===")
students = int(input("Number of students: "))
pages = int(input("Average pages per student: "))

result = calculate_capacity(students, pages)

print(f"\nResults for {students} students with {pages} pages each:")
print(f"  Chunks per student: {result['chunks_per_student']}")
print(f"  Total vectors: {result['total_chunks']:,}")
print(f"  Storage required: {result['storage_gb']:.3f} GB")
print(f"  Initial write units: {result['write_units']:,}")
print(f"  Monthly read units: {result['read_units_monthly']:,}")
print()
print("Free Tier Usage:")
print(f"  Storage: {result['storage_usage_pct']:.1f}%")
print(f"  Write Units: {result['write_usage_pct']:.1f}%")
print(f"  Read Units: {result['read_usage_pct']:.1f}%")
print(f"  Namespaces: {result['namespace_usage_pct']:.1f}%")
print()
if result['within_free_tier']:
    print("✅ Configuration fits within Pinecone free tier!")
else:
    print("❌ Configuration exceeds free tier limits.")
    if result['storage_usage_pct'] > 100:
        print(f"   - Storage: {result['storage_gb']:.2f}GB exceeds 2GB limit")
    if result['write_usage_pct'] > 100:
        print(f"   - Write units: {result['write_units']:,} exceeds 2M limit")
    if result['read_usage_pct'] > 100:
        print(f"   - Read units: {result['read_units_monthly']:,} exceeds 1M limit")
    if result['namespace_usage_pct'] > 100:
        print(f"   - Namespaces: {students} exceeds 100 limit")

## Cost Optimization Tips

1. **Reduce embedding dimensions**: Use 512 or 384 dimensions if accuracy permits
2. **Increase chunk size**: Larger chunks = fewer vectors (but may reduce retrieval quality)
3. **Implement caching**: Cache frequent queries to reduce read units
4. **Batch operations**: Group vector operations to optimize write units
5. **Clean up old data**: Implement retention policies for unused documents

## Usage

Run this notebook to estimate whether your deployment fits within the
[Pinecone free tier](https://www.pinecone.io/pricing/) before going live.

**Quick reference — Pinecone free tier limits:**
| Resource | Free limit |
|----------|------------|
| Storage | 2 GB |
| Write units | 2M / month |
| Read units | 1M / month |
| Namespaces | 100 |
